# 🧩 Models, Prompts and Messages

### Tools and Techniques in Data Science — LangChain Module

| | |
|---|---|
| **Difficulty** | ⭐ Beginner → ⭐⭐ Intermediate |
| **Estimated Time** | 90–120 minutes |
| **Prerequisites** | Notebook 01 (Introduction to LangChain) |

---

**Welcome back!** In this notebook, you'll master the building blocks of LangChain: **models**, **prompts**, and **messages**. These three concepts are the foundation of every LangChain application.

By the end, you'll know how to:
- Engineer effective prompts for data science tasks
- Parse structured data from LLM responses using Pydantic
- Build reusable, parameterized prompt pipelines

> 💡 **Data Science Focus:** Every example uses real data science scenarios — from explaining algorithms to generating code.

## 🎯 Learning Objectives

In this notebook, you will:

1. **Understand** the difference between LLMs and Chat Models
2. **Master** all three message types (System, Human, AI)
3. **Design** effective prompt templates with variables
4. **Apply** few-shot prompting for better data science outputs
5. **Parse** unstructured text into structured data using Pydantic
6. **Build** reusable prompt pipelines for data science tasks
7. **Compare** API and Ollama implementations

---

## ⚙️ Setup

In [ ]:
# Install packages if needed (uncomment)
# !pip install langchain langchain-openai langchain-ollama langchain-text-splitters python-dotenv pydantic

In [ ]:
import os
from dotenv import load_dotenv

# LangChain core
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

# Model integrations
from langchain_openai import ChatOpenAI

# Pydantic for structured output
from pydantic import BaseModel, Field

load_dotenv()

# Verify setup
api_key = os.getenv("OPENAI_API_KEY")
print("OpenAI API key:", "found" if api_key else "NOT SET — Ollama examples will still work")
print("\nAll imports successful!")

---

## 1. Chat Models — Deep Dive

### What is a Chat Model?

A **chat model** is an LLM designed to handle **multi-turn conversations**. Instead of just completing text, it processes a list of messages with roles.

```mermaid
flowchart LR
    A["Chat Model"] --> B["Processes Messages"]
    B --> C["Generates Response"]
```

### LLM vs Chat Model

| | **LLM (Legacy)** | **Chat Model (Current)** |
|---|---|---|
| **Input** | Single text string | List of messages with roles |
| **Interface** | `llm("Complete this text")` | `model.invoke([messages])` |
| **Conversation** | No built-in support | Native multi-turn support |
| **Examples** | text-davinci-003 (deprecated) | GPT-4o, Llama 3.2, Mistral |
| **Status** | Deprecated by most providers | Industry standard |

> 💡 **Key Point:** Always use **Chat Models** in modern LangChain code. The old LLM interface is deprecated.

In [ ]:
# Create a chat model
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Chat models work with lists of messages
response = model.invoke([
    SystemMessage(content="You are a data science expert."),
    HumanMessage(content="What is the difference between L1 and L2 regularization?")
])

print("Response type:", type(response).__name__)
print("\n" + response.content[:500])

### 🔍 What Happened?

- **`temperature=0`** — Makes responses more deterministic (same input → same output)
- **`model.invoke([messages])`** — Sends the message list to the API
- **Returns `AIMessage`** — Contains `.content` (text) and `.response_metadata` (usage info)

### 🧪 Experiment

Try different `temperature` values:
- `0` = Deterministic (same answer every time)
- `0.5` = Balanced creativity
- `1.0` = Maximum creativity (random)

---

## 2. Messages — The Three Types

Messages are the units of communication. Each has a **role** and **content**:

```mermaid
flowchart TD
    S["SystemMessage\n Role: system"] --> M["Model"]
    H["HumanMessage\n Role: user"] --> M
    A["AIMessage\n Role: assistant"] --> M
    M --> R["Response"]
```

| Message Class | Role | Purpose | When to Use |
|---|---|---|---|
| `SystemMessage` | `system` | Sets AI behavior/personality | Always at the start |
| `HumanMessage` | `user` | User's input/question | Each user turn |
| `AIMessage` | `assistant` | AI's previous response | For conversation history |

### SystemMessage — Setting the Expertise Level

The system message is the **most important** message. It sets:
- Who the AI is
- What expertise level to use
- How to format responses
- What topics to focus on

In [ ]:
# System message controls the AI's behavior
system_msgs = [
    ("casual", "You are a friendly tutor. Use simple analogies and humor."),
    ("expert", "You are a senior ML engineer at Google. Give precise, technical answers with code."),
    ("teacher", "You are a university professor. Explain concepts step by step with mathematical notation."),
]

question = "Explain gradient descent in 2 sentences."

for level, system_text in system_msgs:
    response = model.invoke([
        SystemMessage(content=system_text),
        HumanMessage(content=question)
    ])
    print(f"\n--- {level.upper()} ---")
    print(response.content)
    print()

### 🧪 Experiment

Try these system messages:
- `"You are a strict professor who gives short, no-nonsense answers."`
- `"You are a data science coach who encourages students and gives practical tips."`
- `"You respond ONLY in bullet points, never in paragraphs."`

How does each change the response?

---

## 3. Message History — Multi-Turn Conversations

Chat models maintain context through **message history** — a list of all previous messages.

```mermaid
flowchart LR
    subgraph "Message History"
        M1["SystemMessage"] --> M2["HumanMessage"]
        M2 --> M3["AIMessage"]
        M3 --> M4["HumanMessage"]
        M4 --> M5["AIMessage"]
    end
```

### Why It Matters

- The AI **remembers** what you've discussed
- Follow-up questions work naturally
- The AI can build on previous answers

In [ ]:
# Build a conversation history
history = [
    SystemMessage(content="You are a data science tutor. Be concise."),
    HumanMessage(content="What is a confusion matrix?"),
    AIMessage(content="A confusion matrix is a table that shows how well a classification model performs by comparing predicted labels against actual labels. It has four components: True Positives, True Negatives, False Positives, and False Negatives."),
    HumanMessage(content="And how do I calculate precision from it?")
]

response = model.invoke(history)
print("AI:", response.content)
print("\n--- Notice: the AI referenced the confusion matrix discussion above! ---")

### 🔍 What Happened?

The model received **all 4 messages** and understood:
1. Its role (data science tutor)
2. The topic (confusion matrix)
3. What it previously said
4. The follow-up question (precision)

This is how multi-turn conversations work!

> 💡 **Note:** In production, you'd store messages in a database. For this tutorial, we use a Python list.

---

## 4. Prompt Templates

Prompt templates turn your instructions into **reusable, parameterized** formats.

### Why Use Templates?

| Without Template | With Template |
|---|---|
| Hardcoded prompt each time | `{variable}` placeholders |
| Copy-paste everywhere | One template, many uses |
| Error-prone | Consistent formatting |

In [ ]:
# Simple prompt template with one variable
simple_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a data science expert."),
    ("human", "Explain {topic} in simple terms.")
])

# Fill in the variable
messages = simple_prompt.invoke({"topic": "random forest"})
print("Generated messages:")
for msg in messages.messages:
    print(f"  [{msg.type}]: {msg.content}")

# Chain it with a model
chain = simple_prompt | model | StrOutputParser()
result = chain.invoke({"topic": "random forest"})
print("\nResponse:", result[:200])

### Multiple Variables

Templates can have as many `{variables}` as you need:

In [ ]:
# Multi-variable prompt template
expert_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role} with {years} years of experience in {domain}."),
    ("human", "Explain {concept} at a {level} level.")
])

# Different audiences, same template
chain = expert_prompt | model | StrOutputParser()

configs = [
    {"role": "ML engineer", "years": "10", "domain": "healthcare",
     "concept": "XGBoost", "level": "beginner"},
    {"role": "data scientist", "years": "5", "domain": "finance",
     "concept": "XGBoost", "level": "advanced"},
]

for config in configs:
    print(f"\n--- {config['role'].upper()} (years={config['years']}) ---")
    print(chain.invoke(config)[:300])
    print()

---

## 5. Few-Shot Prompting

**Few-shot prompting** gives the AI examples of the input-output pattern you want.

### Why It Works

LLMs learn from examples in the prompt. Providing 2-3 examples dramatically improves consistency.

```mermaid
flowchart TD
    E1["Example 1: Input → Output"] --> T["Template"]
    E2["Example 2: Input → Output"] --> T
    E3["Example 3: Input → Output"] --> T
    Q["Your Question"] --> T
    T --> M["Model"]
    M --> R["Consistent Output"]
```

In [ ]:
# Few-shot prompt: teach the model the format you want
examples = [
    {"concept": "Linear Regression",
     "explanation": "Linear Regression fits a straight line to predict a continuous value from input features. Think of it as finding the best line through a scatter plot."},
    {"concept": "Decision Tree",
     "explanation": "A Decision Tree asks yes/no questions about features to classify data. Like a flowchart that guides you to a decision at each branch."},
]

# Format each example
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "Explain {concept} in one sentence."),
    ("ai", "{explanation}")
])

# Few-shot template combines examples + new question
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

full_prompt = ChatPromptTemplate.from_messages([
    ("system", "You explain ML concepts in exactly one sentence with an analogy."),
    few_shot_prompt,
    ("human", "Explain {concept} in one sentence.")
])

chain = full_prompt | model | StrOutputParser()

# New concept — model follows the pattern from examples
print(chain.invoke({"concept": "Random Forest"}))
print()
print(chain.invoke({"concept": "K-Means Clustering"}))

### 🔍 What Happened?

1. We gave 2 examples of the input→output pattern
2. The model learned: *"Explain ML concepts in one sentence with an analogy"*
3. New concepts follow the **same pattern** automatically

### 🧪 Experiment

Add your own examples and see if the model follows the new pattern!
Try adding an example with a different format (e.g., bullet points) and observe how it changes.

---

## 6. Data Science Prompt Patterns

Here are effective prompt patterns for data science work:

| Pattern | Template | Use Case |
|---|---|---|
| **Explain** | "Explain {concept} at {level} level" | Learning |
| **Code** | "Write Python code to {task} using {library}" | Coding |
| **Debug** | "This code has a bug: {code}. Fix it." | Debugging |
| **Compare** | "Compare {A} vs {B} for {use_case}" | Decision-making |
| **Quiz** | "Generate {n} questions about {topic}" | Studying |

In [ ]:
# Pattern 1: Code Generation
code_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Python expert. Write clean, commented code."),
    ("human", "Write Python code to {task} using {library}. Include comments.")
])

chain = code_prompt | model | StrOutputParser()
print(chain.invoke({
    "task": "calculate precision, recall, and F1-score",
    "library": "sklearn"
}))
print("\n" + "=" * 60)

In [ ]:
# Pattern 2: Quiz Generation
quiz_prompt = ChatPromptTemplate.from_messages([
    ("system", "You create data science quizzes. Format as numbered questions with 4 options (A-D) and the correct answer."),
    ("human", "Generate {n} multiple-choice questions about {topic}.")
])

chain = quiz_prompt | model | StrOutputParser()
print(chain.invoke({"n": 3, "topic": "confusion matrices"}))

In [ ]:
# Pattern 3: Create a Study Plan
plan_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a data science curriculum designer."),
    ("human", "Create a {duration} study plan for {topic}. Include specific resources and milestones.")
])

chain = plan_prompt | model | StrOutputParser()
print(chain.invoke({"duration": "2-week", "topic": "machine learning fundamentals"}))

### 🧪 Experiment

Try creating your own prompt pattern:
```python
# Pattern 4: Error Diagnosis
diagnosis_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a debugging expert."),
    ("human", "I got this error: {error}. My code: {code}. What's wrong and how to fix it?")
])
```

---

## 7. Output Parsing

Output parsers convert the LLM's raw text into usable Python objects.

### The Problem: Unstructured Text

```mermaid
flowchart TD
    P["Prompt"] --> M["Model"]
    M --> T["Unstructured Text"]
    T --> R["Hard to use programmatically"]
```

LLMs return natural language text. You can't do `response.accuracy` on a string!

### The Solution: Structured Output

```mermaid
flowchart TD
    P["Prompt"] --> M["Model"]
    M --> S["Structured Output"]
    S --> A["Python Application"]
    S --> B["response.name"]
    S --> C["response.accuracy"]
```

In [ ]:
# StrOutputParser: extracts plain text from AIMessage
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a data science expert."),
    ("human", "List 3 Python libraries for time series analysis.")
])

# Without parser: returns AIMessage object
raw_response = (prompt | model).invoke({})
print("Raw response type:", type(raw_response).__name__)
print("Content:", raw_response.content[:100])

# With parser: returns plain string
parsed_response = (prompt | model | StrOutputParser()).invoke({})
print("\nParsed response type:", type(parsed_response).__name__)
print("Content:", parsed_response[:100])

### 🔍 What Happened?

- **Without `StrOutputParser`**: Returns `AIMessage` (complex object)
- **With `StrOutputParser`**: Returns plain `str` (easy to use)

`StrOutputParser` is great for text, but what if you need **structured data**?

---

## 8. Structured Output with Pydantic

This is where it gets powerful. Instead of getting text back, you get a **Python object** with typed fields.

### Unstructured Text vs Structured Data

| | **Unstructured Text** | **Structured Data** |
|---|---|---|
| **Format** | Free-form natural language | Typed fields (str, int, list) |
| **Example** | `"Random Forest is an ensemble..."` | `MLConcept(name="Random Forest", ...)` |
| **Programmatic access** | Must parse manually | `concept.name`, `concept.difficulty` |
| **Reliability** | Inconsistent format | Consistent, validated schema |
| **Use case** | Chat, summaries | APIs, databases, dashboards |

### How It Works

1. **Define** a Pydantic model (your schema)
2. **Pass** it to `model.with_structured_output()`
3. **Get** typed Python objects back

In [ ]:
# Define the schema with Pydantic
class MLConcept(BaseModel):
    """A machine learning concept explained for data science students."""

    name: str = Field(description="Name of the ML concept")
    definition: str = Field(description="Clear, concise definition in one sentence")
    intuition: str = Field(description="Intuitive explanation or analogy")
    use_case: str = Field(description="A practical real-world use case")
    difficulty: str = Field(description="Difficulty level: beginner, intermediate, or advanced")

print("Pydantic model defined!")
print("Fields:", list(MLConcept.model_fields.keys()))

In [ ]:
# Create a structured-output model
structured_model = model.with_structured_output(MLConcept)

# Prompt the model
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a data science educator. Explain ML concepts clearly."),
    ("human", "Explain the concept: {concept}")
])

chain = prompt | structured_model
concept = chain.invoke({"concept": "Gradient Boosting"})

# Now you have a REAL Python object!
print("Type:", type(concept).__name__)
print("Name:", concept.name)
print("Definition:", concept.definition)
print("Intuition:", concept.intuition)
print("Use Case:", concept.use_case)
print("Difficulty:", concept.difficulty)

### 🔍 Why This Is Powerful

The response is now a **real Python object** you can use programmatically:

```python
# Access fields directly
print(concept.name)          # "Gradient Boosting"
print(concept.difficulty)     # "intermediate"

# Use in if-statements
if concept.difficulty == "beginner":
    show_in_basics_course(concept)

# Convert to dict for databases
concept_dict = concept.model_dump()  # {'name': 'Gradient Boosting', ...}

# Serialize to JSON for APIs
concept_json = concept.model_dump_json()
```

### 🧪 Experiment

Try explaining different concepts and see how the structured output changes:
- `"XGBoost"`
- `"Neural Networks"`
- `"K-Nearest Neighbors"`

---

## 9. Advanced Pydantic Schemas

You can create more complex schemas for different data science needs:

In [ ]:
# Schema for comparing two ML algorithms
class AlgorithmComparison(BaseModel):
    """Comparison of two machine learning algorithms."""

    algorithm_a: str = Field(description="Name of first algorithm")
    algorithm_b: str = Field(description="Name of second algorithm")
    winner: str = Field(description="Which algorithm is better for the given use case")
    reason: str = Field(description="Why the winner is better")
    tradeoffs: list[str] = Field(description="Key tradeoffs between the two")
    recommendation: str = Field(description="Final recommendation")


# Schema for quiz questions
class QuizQuestion(BaseModel):
    """A multiple-choice quiz question for data science."""

    question: str = Field(description="The quiz question")
    options: list[str] = Field(description="4 answer options")
    correct_answer: str = Field(description="The correct answer")
    explanation: str = Field(description="Why this is the correct answer")


# Schema for study plan
class StudyTopic(BaseModel):
    """A single topic in a study plan."""

    topic: str = Field(description="Topic name")
    duration_hours: int = Field(description="Estimated study hours")
    resources: list[str] = Field(description="Recommended resources")
    milestone: str = Field(description="What you should be able to do after this topic")

print("All schemas defined!")
print("AlgorithmComparison fields:", list(AlgorithmComparison.model_fields.keys()))
print("QuizQuestion fields:", list(QuizQuestion.model_fields.keys()))

In [ ]:
# Generate a structured comparison
structured_model = model.with_structured_output(AlgorithmComparison)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an ML expert. Compare algorithms objectively."),
    ("human", "Compare {algo_a} vs {algo_b} for {use_case}. Which wins?")
])

chain = prompt | structured_model
comparison = chain.invoke({
    "algo_a": "Random Forest",
    "algo_b": "XGBoost",
    "use_case": "predicting customer churn"
})

print(f"Winner: {comparison.winner}")
print(f"Reason: {comparison.reason}")
print(f"\nTradeoffs:")
for t in comparison.tradeoffs:
    print(f"  - {t}")
print(f"\nRecommendation: {comparison.recommendation}")

In [ ]:
# Generate a structured quiz
structured_model = model.with_structured_output(QuizQuestion)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You create data science quizzes."),
    ("human", "Create a quiz question about {topic}. Make one question with 4 options.")
])

chain = prompt | structured_model
quiz = chain.invoke({"topic": "precision vs recall"})

print(f"Question: {quiz.question}")
for i, option in enumerate(quiz.options):
    print(f"  {chr(65+i)}. {option}")
print(f"\nCorrect: {quiz.correct_answer}")
print(f"Explanation: {quiz.explanation}")

---

## 10. Complete API Implementation

Let's build a reusable data science toolkit using OpenAI:

In [ ]:
# Data Science Toolkit using OpenAI API
class DataScienceToolkit:
    """A reusable toolkit for data science tasks using LangChain."""

    def __init__(self, model_name="gpt-4o-mini", temperature=0):
        self.model = ChatOpenAI(model=model_name, temperature=temperature)

    def explain_concept(self, concept: str, level: str = "intermediate") -> MLConcept:
        """Explain an ML concept as a structured object."""
        prompt = ChatPromptTemplate.from_messages([
            ("system", f"You are a data science educator. Explain at {level} level."),
            ("human", "Explain the concept: {concept}")
        ])
        structured = self.model.with_structured_output(MLConcept)
        return (prompt | structured).invoke({"concept": concept})

    def compare_algorithms(self, algo_a: str, algo_b: str, use_case: str) -> AlgorithmComparison:
        """Compare two ML algorithms."""
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are an ML expert. Compare algorithms objectively."),
            ("human", "Compare {algo_a} vs {algo_b} for {use_case}. Which wins?")
        ])
        structured = self.model.with_structured_output(AlgorithmComparison)
        return (prompt | structured).invoke({
            "algo_a": algo_a, "algo_b": algo_b, "use_case": use_case
        })

    def generate_code(self, task: str, library: str = "pandas") -> str:
        """Generate Python code for a data science task."""
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a Python expert. Write clean, commented code."),
            ("human", "Write Python code to {task} using {library}. Include a brief explanation.")
        ])
        return (prompt | self.model | StrOutputParser()).invoke({
            "task": task, "library": library
        })


# Use the toolkit
toolkit = DataScienceToolkit()

# 1. Explain a concept
concept = toolkit.explain_concept("XGBoost", level="beginner")
print(f"Concept: {concept.name}")
print(f"Definition: {concept.definition}")
print(f"Difficulty: {concept.difficulty}")
print()

# 2. Generate code
code = toolkit.generate_code("calculate precision, recall, and F1-score from a confusion matrix")
print("Generated Code:")
print(code)

---

## 11. Ollama Implementation

Same toolkit, running locally with Ollama — no API key needed!

In [ ]:
from langchain_ollama import ChatOllama


class LocalDataScienceToolkit:
    """Data science toolkit using local Ollama models."""

    def __init__(self, model_name="llama3.2", temperature=0):
        self.model = ChatOllama(model=model_name, temperature=temperature)

    def explain_concept(self, concept: str, level: str = "intermediate") -> str:
        """Explain an ML concept."""
        prompt = ChatPromptTemplate.from_messages([
            ("system", f"You are a data science educator. Explain at {level} level."),
            ("human", "Explain the concept: {concept}")
        ])
        return (prompt | self.model | StrOutputParser()).invoke({"concept": concept})

    def generate_code(self, task: str, library: str = "pandas") -> str:
        """Generate Python code."""
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a Python expert. Write clean, commented code."),
            ("human", "Write Python code to {task} using {library}.")
        ])
        return (prompt | self.model | StrOutputParser()).invoke({
            "task": task, "library": library
        })


# Use the local toolkit
try:
    local_toolkit = LocalDataScienceToolkit()
    result = local_toolkit.explain_concept("random forest", level="beginner")
    print("Local Ollama Response:")
    print(result)
except Exception as e:
    print(f"Ollama not available: {e}")
    print("Make sure Ollama is running and you've pulled a model.")

### Side-by-Side Comparison

In [ ]:
# Compare API vs Ollama for the same task
task = "Explain the bias-variance tradeoff in 3 bullet points."

print(f"Task: {task}")
print("=" * 60)

# API version
if os.getenv("OPENAI_API_KEY"):
    api_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a data science expert. Be concise."),
        ("human", "{task}")
    ])
    api_result = (api_prompt | ChatOpenAI(model="gpt-4o-mini") | StrOutputParser()).invoke({"task": task})
    print("\nOpenAI API:")
    print(api_result)
else:
    print("\nOpenAI: not configured")

print("\n" + "=" * 60)

# Ollama version
try:
    ollama_result = (api_prompt | ChatOllama(model="llama3.2") | StrOutputParser()).invoke({"task": task})
    print("\nLocal Ollama:")
    print(ollama_result)
except Exception as e:
    print(f"\nOllama: not available ({e})")

---

## ⚠️ Common Mistakes

| Mistake | Why It's Bad | Fix |
|---|---|---|
| **Vague prompts** | "Tell me about ML" → Generic answer | Be specific: "Explain overfitting for beginners" |
| **No system message** | AI doesn't know its role | Always include a SystemMessage |
| **Forgetting temperature** | Inconsistent outputs | Set `temperature=0` for deterministic tasks |
| **Not parsing output** | Hard to use in code | Use `StrOutputParser` or `with_structured_output` |
| **Too few examples** | Inconsistent format in few-shot | Give 2-3 clear examples |
| **Ignoring token limits** | Errors or truncated responses | Keep inputs concise, watch context length |
| **Hardcoding API keys** | Security risk | Always use `.env` + `python-dotenv` |

In [ ]:
# Example of bad vs good prompting

# BAD: Vague prompt
bad_prompt = ChatPromptTemplate.from_messages([
    ("human", "Tell me about ML")
])

# GOOD: Specific prompt with context
good_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a data science tutor for university students. Be concise and use examples."),
    ("human", "Explain the difference between supervised and unsupervised learning. Give 2 real-world examples of each.")
])

print("BAD prompt output:")
print("-" * 40)
if os.getenv("OPENAI_API_KEY"):
    print((bad_prompt | model | StrOutputParser()).invoke({})[:200])

print("\nGOOD prompt output:")
print("-" * 40)
if os.getenv("OPENAI_API_KEY"):
    print((good_prompt | model | StrOutputParser()).invoke({})[:300])

---

## 🏋️ Exercises

Complete these exercises to solidify your understanding.

### Exercise 1: Build a Data Science Terminology Dictionary

Create a Pydantic model `TermDefinition` with fields:
- `term`: The data science term
- `definition`: A one-sentence definition
- `category`: One of "statistics", "ML", "programming", or "data engineering"
- `related_terms`: A list of related terms

Then build a chain that generates a structured definition for any term.

In [ ]:
# Exercise 1: Your code here!
#
# Steps:
# 1. Define a Pydantic model:
#    class TermDefinition(BaseModel):
#        term: str
#        definition: str
#        category: str
#        related_terms: list[str]
#
# 2. Create a prompt + structured model chain
# 3. Test with: "gradient descent", "cross-validation", "feature engineering"


### Exercise 2: Few-Shot Data Cleaner

Create a few-shot prompt that takes messy column names and outputs clean ones:

| Input | Output |
|---|---|
| `"First Name"` | `"first_name"` |
| `"Annual Income ($)"` | `"annual_income_usd"` |
| `"Customer ID #"` | `"customer_id"` |

Give 3 examples and test with new column names!

In [ ]:
# Exercise 2: Your code here!
#
# Steps:
# 1. Create example pairs (input → output)
# 2. Build a FewShotChatMessagePromptTemplate
# 3. Create a chain and test with:
#    - "Date of Birth"
#    - "Total Revenue (USD)"
#    - "Email Address"


### Exercise 3: Multi-Format Explainer

Create a prompt that takes an ML concept and explains it in three formats:
1. **For a 10-year-old** (simple analogies)
2. **For a professor** (mathematical details)
3. **For a CEO** (business impact only)

Use the same model but different system messages for each format.

In [ ]:
# Exercise 3: Your code here!
#
# Steps:
# 1. Define 3 system messages (child, professor, CEO)
# 2. Create a chain that takes a concept and format level
# 3. Test with "neural networks" at all 3 levels


### 🌟 Challenge: Data Science Exam Generator

Build a complete exam generator that:

1. Takes a topic and number of questions
2. Generates a mix of question types:
   - Multiple choice (QuizQuestion schema)
   - Short answer questions
3. Includes an answer key
4. Provides difficulty ratings for each question

**Bonus:** Output the exam as a structured JSON that could be loaded into a quiz app!

In [ ]:
# 🌟 Challenge: Your code here!
#
# Suggested Pydantic schema:
# class ExamQuestion(BaseModel):
#     question: str
#     question_type: str  # "multiple_choice" or "short_answer"
#     options: list[str] | None  # for multiple choice
#     answer: str
#     explanation: str
#     difficulty: str  # "easy", "medium", "hard"
#
# class Exam(BaseModel):
#     topic: str
#     questions: list[ExamQuestion]
#     total_points: int


---

## 📝 Key Takeaways

| Concept | What It Is | Key Insight |
|---|---|---|
| **Chat Model** | LLM that handles conversations | Always use Chat Models (not legacy LLMs) |
| **SystemMessage** | Sets AI behavior | Most important message — write it carefully |
| **HumanMessage** | User input | Be specific and clear |
| **AIMessage** | AI response | Used for conversation history |
| **Prompt Template** | Reusable format with `{variables}` | One template → many uses |
| **Few-Shot** | Examples in the prompt | 2-3 examples dramatically improve consistency |
| **StrOutputParser** | Extracts text from AIMessage | Use for plain text responses |
| **Structured Output** | Pydantic + `with_structured_output()` | Get typed Python objects back |

### The Two Pipelines

```python
# Text output pipeline
text_chain = prompt | model | StrOutputParser()
result = text_chain.invoke({"topic": "XGBoost"})  # Returns str

# Structured output pipeline
structured_model = model.with_structured_output(MLConcept)
structured_chain = prompt | structured_model
result = structured_chain.invoke({"concept": "XGBoost"})  # Returns MLConcept
print(result.name)  # Access fields directly!
```

### Best Practices

1. **Always** include a SystemMessage
2. **Be specific** in your prompts
3. **Use few-shot** for format consistency
4. **Use structured output** when you need programmatic access
5. **Set temperature=0** for deterministic tasks
6. **Never** hardcode API keys

---

## 🚀 Next Steps

| Notebook | Topic | What You'll Learn |
|---|---|---|
| **01** | Introduction | What is LangChain? (completed) |
| **02** | Models, Prompts & Messages | You are here! |
| **03** | LCEL & Chains | Build complex multi-step pipelines |
| **04** | Embeddings & Vector Stores | Semantic search and embeddings |
| **05** | RAG Applications | Question answering over documents |
| **06** | Tools & Agents | AI that can take actions |
| **07** | Advanced Project | Build a complete data science assistant |

---

🎉 **Great work!** You've mastered models, prompts, and messages.

Next up: **LCEL & Chains** — where you'll learn to build complex, multi-step pipelines! 🚀